In [ ]:
from pathlib import Path
import sys

# Go up one directory to the project root
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier as KNC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve,roc_auc_score

RANDOM_SEED = 2566


In [ ]:
from src.analysis import read_data
from src.models import *

In [ ]:
awake_path='../data/awake_data.csv'
sedated_path='../data/sedate_data.csv'
awake_df,sedate_df =read_data(awake_path,sedated_path)

**subject-wise splitting**

In [ ]:
id_list=[1067,1068,1071,1074]

In [ ]:
awake_train=awake_df.loc[~awake_df['subject'].isin(id_list)]
sedate_train=sedate_df.loc[~sedate_df['subject'].isin(id_list)]
awake_test=awake_df.loc[awake_df['subject'].isin(id_list)]
sedate_test=sedate_df.loc[sedate_df['subject'].isin(id_list)]

In [ ]:
awake_train.shape,sedate_train.shape
awake_test.shape,sedate_test.shape

In [ ]:
train_df = pd.concat([awake_train,sedate_train],ignore_index=True)
test_df = pd.concat([awake_test,sedate_test],ignore_index=True)

In [ ]:
train_df.hist(figsize=(10,10))

**label encoding**

In [ ]:
y_train = train_df['label']
y_test= test_df['label']
label_value = {'sedated':0,'awake':1}
y_train=y_train.map(label_value)
y_test=y_test.map(label_value)

**spectral features**

In [ ]:
X_train_spectral= train_df.drop(['subject','label','MI','o-info','complexity'],axis=1)
X_test_spectral =  test_df.drop(['subject','label','MI','o-info','complexity'],axis=1)

In [ ]:
X_train_scaled_spectral,X_test_scaled_spectral=scale(X_train_spectral,X_test_spectral)

In [ ]:

forest_clf = RFC(random_state=RANDOM_SEED)
xgb_clf = XGBClassifier(random_state=RANDOM_SEED)
logistic_clf = LogisticRegression()
svm_clf = SVC(random_state=RANDOM_SEED, probability=True)
knn_clf = KNC()

In [ ]:
train_feature(forest_clf,X_train_scaled_spectral,X_test_scaled_spectral,y_train,y_test)

In [ ]:
auc(forest_clf,X_test_scaled_spectral,y_test)

In [ ]:
train_feature(xgb_clf,X_train_scaled_spectral,X_test_scaled_spectral,y_train,y_test)

In [ ]:
auc(xgb_clf,X_test_scaled_spectral,y_test)

In [ ]:
train_feature(logistic_clf,X_train_scaled_spectral,X_test_scaled_spectral,y_train,y_test)

In [ ]:
auc(logistic_clf,X_test_scaled_spectral,y_test)

In [ ]:
train_feature(svm_clf,X_train_scaled_spectral,X_test_scaled_spectral,y_train,y_test)

In [ ]:
auc(svm_clf,X_test_scaled_spectral,y_test)

In [ ]:
train_feature(knn_clf,X_train_scaled_spectral,X_test_scaled_spectral,y_train,y_test)

In [ ]:
auc(knn_clf,X_test_scaled_spectral,y_test)

**iit features**

In [ ]:
X_train_iit= train_df.drop(['subject','label','alpha','beta','theta','delta'],axis=1)
X_test_iit =  test_df.drop(['subject','label','alpha','beta','theta','delta'],axis=1)

In [ ]:
X_train_iit.hist(figsize=(8,8))

In [ ]:
X_train_iit_svm = X_train_iit.copy()
X_test_iit_svm = X_test_iit.copy()

In [ ]:
X_train_iit_svm['o-info'] = np.log(X_train_iit_svm['o-info']+3)
X_train_iit_svm['MI']=np.log(X_train_iit_svm['MI']+1)
X_train_iit_svm['complexity']=X_train_iit_svm['complexity']**2

In [ ]:
X_train_iit_svm.hist()

In [ ]:
X_test_iit_svm['o-info'] = np.log(X_test_iit_svm['o-info']+3)
X_test_iit_svm['MI']=np.log(X_test_iit_svm['MI']+1)
X_test_iit_svm['complexity']=X_test_iit_svm['complexity']**2

In [ ]:
X_train_scaled_iit,X_test_scaled_iit=scale(X_train_iit,X_test_iit)
X_train_scaled_iit_svm,X_test_scaled_iit_svm=scale(X_train_iit_svm,X_test_iit_svm)

In [ ]:
forest_clf_iit = RFC(random_state=RANDOM_SEED)
svm_clf_iit = SVC(random_state=RANDOM_SEED, probability=True)

In [ ]:
train_feature(forest_clf_iit,X_train_scaled_iit,X_test_scaled_iit,y_train,y_test)

In [ ]:
auc(forest_clf_iit,X_test_scaled_iit,y_test)

In [ ]:
train_feature(svm_clf_iit,X_train_scaled_iit_svm,X_test_scaled_iit_svm,y_train,y_test)

In [ ]:
auc(svm_clf_iit,X_test_scaled_iit_svm,y_test)

**spectral+iit**

**spectral+MI**

In [ ]:
X_train_MI= train_df.drop(['subject','label','o-info','complexity'],axis=1)
X_test_MI=  test_df.drop(['subject','label','o-info','complexity'],axis=1)
X_train_MI_scaled,X_test_MI_scaled=scale(X_train_MI,X_test_MI)

In [ ]:
X_train_MI_svm = X_train_MI.copy()
X_test_MI_svm = X_test_MI.copy()
X_train_MI_svm['MI'] = np.log(X_train_MI_svm['MI']+1)
X_test_MI_svm['MI'] = np.log(X_test_MI_svm['MI']+1)
X_train_MI_scaled_svm,X_test_MI_scaled_svm=scale(X_train_MI_svm,X_test_MI_svm)

In [ ]:
forest_MI = RFC(random_state=RANDOM_SEED)
svm_MI = SVC(random_state=RANDOM_SEED, probability=True)

In [ ]:
train_feature(forest_MI,X_train_MI_scaled,X_test_MI_scaled,y_train,y_test)

In [ ]:
auc(forest_MI,X_test_MI_scaled,y_test)

In [ ]:
train_feature(svm_MI,X_train_MI_scaled_svm,X_test_MI_scaled_svm,y_train,y_test)

In [ ]:
auc(svm_MI,X_test_MI_scaled_svm,y_test)

In [ ]:
shap_plot(forest_MI,X_train_MI_scaled,X_test_MI_scaled,X_train_MI)

**spectral+complexity**

In [ ]:
X_train_com= train_df.drop(['subject','label','o-info','MI'],axis=1)
X_test_com=  test_df.drop(['subject','label','o-info','MI'],axis=1)
X_train_com_scaled,X_test_com_scaled=scale(X_train_com,X_test_com)

In [ ]:
X_train_com_svm = X_train_com.copy()
X_test_com_svm = X_test_com.copy()
X_train_com_svm['complexity'] =X_train_com_svm['complexity']**2
X_test_com_svm['complexity'] = X_test_com_svm['complexity']**2
X_train_com_scaled_svm,X_test_com_scaled_svm=scale(X_train_com_svm,X_test_com_svm)

In [ ]:
forest_com = RFC(random_state=RANDOM_SEED)
svm_com = SVC(random_state=RANDOM_SEED, probability=True)

In [ ]:
train_feature(forest_com,X_train_com_scaled,X_test_com_scaled,y_train,y_test)

In [ ]:
auc(forest_com,X_test_com_scaled,y_test)

In [ ]:
train_feature(svm_com,X_train_com_scaled_svm,X_test_com_scaled_svm,y_train,y_test)

In [ ]:
auc(svm_com,X_test_com_scaled_svm,y_test)

In [ ]:
shap_plot(forest_com,X_train_com_scaled,X_test_com_scaled,X_train_com)

**spectral+o-info**

In [ ]:
X_train_o= train_df.drop(['subject','label','MI','complexity'],axis=1)
X_test_o=  test_df.drop(['subject','label','MI','complexity'],axis=1)
X_train_o_scaled,X_test_o_scaled=scale(X_train_o,X_test_o)


In [ ]:
X_train_o_svm = X_train_o.copy()
X_test_o_svm = X_test_o.copy()
X_train_o_svm['o-info'] = np.log(X_train_o_svm['o-info']+3)
X_test_o_svm['o-info'] = np.log(X_test_o_svm['o-info']+3)
X_train_o_scaled_svm,X_test_o_scaled_svm=scale(X_train_o_svm,X_test_o_svm)

In [ ]:
forest_o = RFC(random_state=RANDOM_SEED)
svm_o= SVC(random_state=RANDOM_SEED, probability=True)

In [ ]:
train_feature(forest_o,X_train_o_scaled,X_test_o_scaled,y_train,y_test)

In [ ]:
auc(forest_o,X_test_o_scaled,y_test)

In [ ]:
train_feature(svm_o,X_train_o_scaled_svm,X_test_o_scaled_svm,y_train,y_test)

In [ ]:
auc(svm_o,X_test_o_scaled_svm,y_test)

In [ ]:
shap_plot(forest_o,X_train_o_scaled,X_test_o_scaled,X_train_o)

**combine everything**

In [ ]:
X_train = train_df.drop(['subject','label'],axis=1)
X_test =  test_df.drop(['subject','label'],axis=1)


In [ ]:
X_train_scaled,X_test_scaled=scale(X_train,X_test)

In [ ]:
forest_clf_combined = RFC(random_state=RANDOM_SEED)

In [ ]:
train_feature(forest_clf_combined,X_train_scaled,X_test_scaled,y_train,y_test)

**fine-tune best model**

In [ ]:
fig,ax = plt.subplots(figsize=(8, 3))


clf_params = {'n_estimators': [10, 50, 100, 200],
              'max_depth': [3, 5, None],
              'ccp_alpha': [0, 0.01, 0.1]
              }

cv_splitter = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)


tuned_forest_MI = GridSearchCV(estimator=forest_MI, param_grid=clf_params,
                             scoring=['accuracy'], refit='accuracy', cv=cv_splitter)




In [ ]:
train_feature(tuned_forest_MI,X_train_MI_scaled,X_test_MI_scaled,y_train,y_test)

In [ ]:
print('Best hyperparameters: {}'.format(tuned_forest_MI.best_params_))
print("Best cross-validation score: {:.2f}".format(tuned_forest_MI.best_score_))